# 05 Temporal structure (Phase 5)
No sequence dataset is created. Summaries come from `python -I src/audit/temporal.py`; the figure below reads the disposable timeline cache.

In [1]:
import sys, json
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
R = ROOT / "results/audit"
M = ROOT / "data/manifests"
pd.set_option("display.max_rows", 80); pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200)
print("project root:", ROOT.name)

project root: research_paper


In [2]:
TF = json.loads((R / 'temporal_feasibility.json').read_text(encoding='utf-8'))
rows = []
for f, v in TF['per_file'].items():
    if v['variant'] != 'improved': continue
    rows.append({'file': f.split('/')[-1], **{k: v[k] for k in ['n_flows','share_bins_empty','longest_empty_run_bins','flows_per_nonempty_bin_p50','file_order_backward_steps_by_start','file_order_backward_steps_by_end','n_duration_gt_window','bins_with_attack_excl_attempted']}})
pd.DataFrame(rows)

,file,n_flows,share_bins_empty,longest_empty_run_bins,flows_per_nonempty_bin_p50,file_order_backward_steps_by_start,file_order_backward_steps_by_end,n_duration_gt_window,bins_with_attack_excl_attempted
0,friday.csv,547557,0.177896,19,7.0,248931,263477,168088,1366
1,monday.csv,371624,0.115176,11,10.0,174479,174067,108524,0
2,thursday.csv,362076,0.199396,20,6.0,165271,165027,89074,557
3,tuesday.csv,322078,0.153588,17,7.0,152339,152137,97598,2652
4,wednesday.csv,496641,0.178433,29,7.0,196756,225855,118730,1148
5,Friday-02-03-2018.csv,6311371,0.060077,17,344.0,2863328,2865309,2401568,10193
6,Friday-16-02-2018.csv,7390266,0.026167,64,324.0,2540804,3353277,1651550,389
7,Friday-23-02-2018.csv,5976481,0.801694,42995,301.0,2721984,2726422,2168107,158
8,Thursday-01-03-2018.csv,6551401,0.030909,12,344.0,2981618,2984080,2468271,1249
9,Thursday-15-02-2018.csv,5410102,0.076800,11,267.0,2439689,2444788,1674792,1543


In [3]:
SEG = pd.read_csv(R / 'attack_segments.csv')
SEG[SEG.gap_threshold_s == 60].groupby('dataset_family').agg(n_segments=('segment','count'), median_lead_s=('lead_since_prev_attack_or_file_start_s','median'))

,n_segments,median_lead_s
dataset_family,,
CIC-IDS-2017,98,120.255
CSE-CIC-IDS-2018,285,120.314


In [4]:
import duckdb, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from src.audit.integrity import timeline_path
from src.audit.common import DataFile
f = DataFile('CIC-IDS-2017', 'improved', 'data/raw_improved/CIC-IDS-2017/wednesday.csv', ROOT / 'data/raw_improved/CIC-IDS-2017/wednesday.csv')
p = timeline_path(f)
if p.exists():
    d = duckdb.sql(f"SELECT date_trunc('minute', ts) m, count(*) n, count_if(upper(label) <> 'BENIGN' AND label NOT LIKE '% - Attempted') a FROM read_parquet('{p.as_posix()}') GROUP BY 1 ORDER BY 1").df()
    fig, ax = plt.subplots(figsize=(11, 3)); ax.plot(d.m, d.n, lw=.8, label='all flows / min'); ax.plot(d.m, d.a, lw=.8, label='attack flows / min'); ax.set_yscale('symlog'); ax.legend(); ax.set_title('CIC-IDS-2017 improved, Wednesday (UTC)')
    out = R / 'figures'; out.mkdir(exist_ok=True); fp = out / 'wednesday_2017_flows_per_minute.png'
    if fp.exists():
        print('figure exists, not overwritten:', fp.name)
    else:
        fig.savefig(fp, dpi=120, bbox_inches='tight'); print('saved figure', fp.name)
else:
    print('timeline cache not present; run src/audit/integrity.py timeline')

figure exists, not overwritten: wednesday_2017_flows_per_minute.png
